In [ ]:
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torch.optim import Adam, AdamW
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)



In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)



In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
sample_image, sample_label = train_dataset[0]

print(f"\n Image shape: {sample_image.shape}")
print(f"Label: {sample_label}")


In [ ]:
# Get one batch of images and labels
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # TODO: What are the dimensions of the first layer? (hint: features? hidden neurons?)
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)

        # activation function for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # Layer 1: TODO
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        # Layer 2 TODO
        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        # Layer 3 TODO
        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        # Output layer TODO
        z4 = self.layer4(a3)

        # TODO: are we missing output activation function? (hint: Loss Function)

        return z4

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        # TODO: make predictions
        outputs = model(X_batch) # shape: (batch_size, 10)
        # TODO: compute loss
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            # TODO: make predictions
            outputs = model(X_batch)  # shape: (batch_size, 10)
            # TODO: compute loss
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # TODO: Apply Softmax to get probabilities
            probabilities = F.softmax(outputs, dim=1)

            # TODO: Multiclass predictions
            predicted = torch.argmax(probabilities, dim=1)

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    # TODO: how do we calculate accuracy?
    accuracy = correct / total

    return avg_loss, accuracy


In [ ]:
N_SAMPLES = 10
indices = np.random.choice(len(train_dataset), N_SAMPLES, replace=False)

# Extract images and labels
images = []
labels = []

for idx in indices:
  img, label = train_dataset[idx]
  img = img.numpy().flatten() # Flatten makes each 28x28 image into 784D
  images.append(img)
  labels.append(label)

X = np.array(images)
y = np.array(labels)

print(f"Data shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique labels: {np.unique(y)}")

In [ ]:
# Task 4: Define device, model, loss, optimizer:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = 3888

hidden_dim = 32          # Design choice

output_dim = 9

model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

# TODO: Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), learning_rate)

# Run Training
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:

# Set model to evaluation mode
model.eval()
# Get one batch from the test DataLoader
images, labels = next(iter(test_loader))
# Move images to device
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    # Flatten images before passing to the model
    outputs = model(images.view(images.size(0), -1))
    predictions = torch.argmax(outputs, dim=1)

# Move tensors back to CPU for plotting
images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

# Plot first 6 predictions
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].squeeze())
    plt.title(f"True: {labels[i]} | Pred: {predictions[i]}")
    plt.axis('off')

plt.tight_layout()
plt.show()